# Demo: Snapshot and eBPF Stage

This notebook demonstrates saving/loading an IdentityHopGraph snapshot (SQLite) and running a sample eBPF event through the `ebpf_analysis_stage`.

In [ ]:
# 1. Save a snapshot to demo_snapshot.db
from src.core.graph.identity_hopgraph import GLOBAL_IDENTITY_GRAPH
GLOBAL_IDENTITY_GRAPH.add_edge('user:alice', 'host:vm1', 'login', 0.5)
GLOBAL_IDENTITY_GRAPH.save_snapshot('demo_snapshot.db')
print('Saved snapshot: demo_snapshot.db')

In [ ]:
# 2. Load the snapshot into a fresh graph instance
from src.core.graph.identity_hopgraph import IdentityHopGraph
g = IdentityHopGraph()
ok = g.load_snapshot('demo_snapshot.db')
print('Loaded snapshot:', ok, 'edges:', len(g._adj))

In [ ]:
# 3. Run a sample eBPF event through the stage
from src.core.event_pipeline.stages.ebpf_analysis import ebpf_analysis_stage
from src.core.event_pipeline.stages.base import StageContext
import asyncio
ev = {'source':'falco_ebpf','command':'/bin/sh -c nsenter --mount /proc/1/ns/mnt','container_id':'cid-demo','syscall':'execve','rule_name':'Container Escape Detected'}
ctx = StageContext(registry=None, config=None, logger=None, state={})
loop = asyncio.new_event_loop()
res = loop.run_until_complete(ebpf_analysis_stage(ev, ctx))
loop.close()
print('eBPF factors:', res.factors)